# Training a real model with VPO: MuSiQue + LoRA

<a href="https://colab.research.google.com/github/ryanboldi/vpo/blob/main/notebooks/03_train_musique_lora.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Notebook 01](01_vpo_from_scratch.ipynb) built VPO on a toy problem in NumPy. This notebook does the
real thing, end to end:

1. Look at the **MuSiQue** task (multi-hop question answering) and its 5-number verifier.
2. Train **Qwen3-1.7B** on it twice with **LoRA** on a single GPU: once with **GRPO**, once with **VPO**.
3. Evaluate both (plus the untrained base model) on held-out questions.
4. Reproduce the paper's **best@k** figure: answer quality of the best of k tries, as a function of k.

**Hardware and time.** Everything here was verified on a single RTX 5090 (32 GB). Budget roughly 3.5 h
for the VPO run, 2.5 h for the GRPO run, and 15 min per evaluation. A 24 GB card should work with the
same settings; a 16 GB card (Colab T4) will not fit this model, so on Colab pick an A100 or L4 runtime
and expect the smaller card to need a smaller model. The training cells are plain shell commands, so you
can also run them in a terminal and come back to the notebook for the analysis.

**What we mean by reproduce.** The paper's MuSiQue run trains Qwen3-1.7B on the full 19,938-question
training set with batch size 128 and 8 rollouts per prompt on 4 H100s. This notebook is the same
pipeline with smaller dials: 2,048 training questions, batch 32, 4 rollouts, LoRA instead of full
fine-tuning, one consumer GPU. The point is to watch the same qualitative result appear: VPO matching
GRPO at k = 1 while pulling ahead as k grows.

## 1. Setup

Clone the repo and install the vendored, patched veRL (it carries the VPO advantage estimator) plus the
`vpo` package. If you already work inside the repo with everything installed, the cell just verifies the
imports.

In [ ]:
import os, sys

# On Colab or a fresh machine, uncomment to clone and install (takes a few minutes):
# !git clone https://github.com/ryanboldi/vpo.git
# %cd vpo
# !cd verl && pip install -q -e . && cd ..
# !pip install -q -e .

# Make sure we run from the repo root, with the vendored veRL first on the path.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'verl'); sys.path.insert(0, '.')
os.environ['PYTHONPATH'] = os.getcwd() + '/verl:' + os.getcwd() + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')

from verl.trainer.ppo.core_algos import AdvantageEstimator as AE
print('veRL advantage estimators include:', [AE[n].value for n in ['VPO', 'VPO_SINGLE', 'GRPO']])

## 2. The task: MuSiQue

MuSiQue is a multi-hop question answering dataset. Each question needs a **chain** of facts: to answer
"When was the team that Khari Long played for founded?" you first find who Khari Long played for, then
find when that team was founded. The prompt shows the model **20 paragraphs** (the 2 to 4 gold ones that
contain the chain, plus distractors), and the model must do two things:

- cite the paragraphs that support its reasoning, inside `<support> ... </support>`
- give the final answer, inside `<answer> ... </answer>`

The cell below downloads the dataset, builds train and test files (we cap them at 2,048 train and 256
test questions for this smoke-scale run), and prints one example.

In [ ]:
!python data/preprocess_musique.py --local_save_dir ~/data/musique_smoke --max_train 2048 --max_test 256

import datasets
ds_test = datasets.Dataset.from_parquet(os.path.expanduser('~/data/musique_smoke/test.parquet'))
row = ds_test[0]
gt = row['reward_model']['ground_truth']
print(row['prompt'][0]['content'][:900])
print('  ...', len(row['prompt'][0]['content']), 'characters total ...')
print()
print('gold answer:   ', gt['answer'])
print('gold support:  ', gt['supporting_indices_ordered'], '(one paragraph per reasoning step, in order)')

## 3. The verifier: a 5-number score vector

A response to a MuSiQue question can be good in five separate ways, so the verifier returns five
numbers, not one:

| dim | name | meaning |
|---|---|---|
| 0 | `hop_1` | did `<support>` cite the gold paragraph for reasoning step 1? (0 or 1) |
| 1 | `hop_2` | same for step 2 |
| 2 | `hop_3` | same for step 3; automatically 1 if the question has fewer than 3 hops |
| 3 | `hop_4` | same for step 4; automatically 1 if fewer than 4 hops |
| 4 | `answer_f1` | word-overlap F1 between `<answer>` and the gold answer, between 0 and 1 |

F1 in one sentence: split both answers into words, and score high when the predicted words and the gold
words mostly overlap, so "the year 1960" against gold "1960" still gets credit.

When GRPO needs a single number it uses a weighted average of the five, with `answer_f1` counted three
times, so `(hop_1 + hop_2 + hop_3 + hop_4 + 3 * answer_f1) / 7`.

This is the exact reward code the trainer calls. Let's run it on two hand-written responses for the
example above:

In [ ]:
from vpo_tasks.musique import TASK

support = ', '.join(str(i) for i in gt['supporting_indices_ordered'])
good = f"The chain is in paragraphs {support}. <support>{support}</support> <answer>{gt['answer']}</answer>"
bad  = "<support>0, 1</support> <answer>purple monkeys</answer>"

print('a correct response:', good[:110], '...')
for name, resp in [('good', good), ('bad ', bad)]:
    out = TASK.score_one(resp, gt, None)
    print(f"{name} response -> vector {[round(v, 2) for v in out['sub_scores']]}   GRPO scalar {out['scalar']:.3f}")

## 4. What changes for VPO: m answers per response

As in notebook 01, VPO asks the model for **several answers in one response**. At training time a small
prompt rewrite replaces the output instructions with a request for m = 3 attempts in
`<response_i> ... </response_i>` tags. The verifier then scores each attempt separately, giving a
**3 x 5 score matrix** per response, and the VPO advantage scores that matrix as a set: draw random
preference weights over the 5 objectives, let each weighting pick its favorite attempt, and average.

GRPO never sees the matrix. It trains on single-answer responses scored by the weighted scalar above.

Here is the actual rewrite, and the actual matrix scoring, on fake text:

In [ ]:
tail = ("First reason about which paragraphs are relevant, then output your supporting paragraph "
        "indices and answer.\n<support>comma-separated paragraph indices (e.g., 3, 7, 12)</support>\n"
        "<answer>your answer</answer>")
print('single-answer instruction tail:\n ', tail.splitlines()[-2], '...')
print()
print('after the multi-solution rewrite:\n ', TASK.rewrite_multi_solution(tail, 3))

first_hop = gt['supporting_indices_ordered'][0]
fake_multi = f"""
<response_1><support>{support}</support><answer>{gt['answer']}</answer></response_1>
<response_2><support>{support}</support><answer>not sure, maybe {gt['answer']} or later</answer></response_2>
<response_3><support>{first_hop}</support><answer>purple monkeys</answer></response_3>
"""
out = TASK.score_multi(fake_multi, 3, gt, None)
print()
print('score matrix (3 attempts x 5 objectives):')
for r in out['sub_scores']:
    print('  ', [round(v, 2) for v in r])

## 5. LoRA in one paragraph

Fine-tuning all 1.7 billion weights with RL needs optimizer state and gradients for every weight, which
does not fit comfortably on one consumer GPU. **LoRA** (Low-Rank Adaptation) freezes the base model and
adds a small trainable correction to each linear layer: instead of updating the big weight matrix `W`,
it learns two thin matrices `A` and `B` (rank 32 here) whose product `B A` is added to `W`. Only `A` and
`B` train. In this run that is about 35 M trainable parameters, roughly 2 percent of the model, and the
saved adapter is 70 MB instead of 4 GB. veRL handles the plumbing: the trainer wraps the model with the
adapters, and the vLLM engine that generates rollouts gets the updated adapter weights after every step.

One honest caveat: LoRA constrains what the update can express, so headline numbers land below the
paper's full fine-tune. The comparison we care about (GRPO versus VPO, same budget, same adapter) is
unaffected.

## 6. Train: one command per method

`train.sh` wires everything: the dataset, the reward function (`vpo/reward.py`, which dispatches to the
MuSiQue verifier we just used), the advantage estimator, and the multi-answer prompt rewrite for VPO.
The two runs below differ **only** in `METHOD`. Everything downstream of that one variable is what this
repo (and the paper) is about:

- `METHOD=vpo` sets `algorithm.adv_estimator=vpo` and turns on the m = 3 rewrite.
- `METHOD=grpo` sets `algorithm.adv_estimator=grpo` and leaves prompts single-answer.

The flags shared by both runs:

| flag | value | why |
|---|---|---|
| `MODEL` | Qwen/Qwen3-1.7B | the paper's MuSiQue model |
| `N_GPUS=1` | batch 32, 4 rollouts per prompt | single-GPU preset; the 4 rollouts form the group that advantages are standardized within |
| `EPOCHS=1` | 2,048 questions / batch 32, about 64 steps | smoke scale (a few overlong prompts get filtered, so the exact step count can be one less) |
| `lora_rank=32, lora_alpha=32` | LoRA on every linear layer | fits the 5090 |
| `rollout.load_format=safetensors` | | required so vLLM can host the LoRA adapter |
| `test_freq=16` | validate every 16 steps | gives us learning curves |
| `save_freq=32` | checkpoint mid-run and at the end | the final one is what we evaluate |

After training, `train.sh` merges the LoRA adapter into the base weights and writes a standard
HuggingFace folder at `.../global_step_<last>/actor/huggingface_merged`, which the evaluation loads
directly.

Each cell streams the trainer console. Expect about 3 minutes per step for VPO and 2 for GRPO.

In [ ]:
%%time
!MUSIQUE_DATA=$HOME/data/musique_smoke METHOD=vpo TASK=musique MODEL=Qwen/Qwen3-1.7B \
  N_GPUS=1 EPOCHS=1 SEED=0 TAG=smoke bash train.sh \
  actor_rollout_ref.model.lora_rank=32 \
  actor_rollout_ref.model.lora_alpha=32 \
  actor_rollout_ref.rollout.load_format=safetensors \
  'trainer.logger=[console]' \
  trainer.save_freq=32 \
  trainer.test_freq=16 2>&1 | grep --line-buffered -oE "step:[0-9]+ - [^[]{0,100}|Merging.*"

In [ ]:
%%time
!MUSIQUE_DATA=$HOME/data/musique_smoke METHOD=grpo TASK=musique MODEL=Qwen/Qwen3-1.7B \
  N_GPUS=1 EPOCHS=1 SEED=0 TAG=smoke bash train.sh \
  actor_rollout_ref.model.lora_rank=32 \
  actor_rollout_ref.model.lora_alpha=32 \
  actor_rollout_ref.rollout.load_format=safetensors \
  'trainer.logger=[console]' \
  trainer.save_freq=32 \
  trainer.test_freq=16 2>&1 | grep --line-buffered -oE "step:[0-9]+ - [^[]{0,100}|Merging.*"

## 7. What happened during training

The trainer logs one line per step. We pull two things out of each log:

- **the training reward** (`critic/score/mean`). Careful reading this across methods: GRPO's score is
  the weighted scalar of one answer, while VPO's is summed over the m = 3 attempts in the response, so
  the two curves live on different scales. What matters is that each one goes up.
- **VPO's own objective** (`vpo/own_pool_expected_max_mean`): the average over random preference
  weightings of the best attempt in the response. This is the set-quality number VPO is actually
  optimizing, and the one to watch on the VPO run.

Every 16 steps the trainer also runs **validation** on the held-out questions (3 sampled responses
each), which gives a small generalization curve: `val-core/musique/reward/best@3/mean` is the best of
the 3 responses' rewards, averaged over questions.

In [ ]:
import re
import matplotlib.pyplot as plt

def parse_log(path, keys):
    out = {k: [] for k in keys}
    steps = {k: [] for k in keys}
    for line in open(path, errors='ignore'):
        m = re.search(r'step:(\d+) - ', line)
        if not m:
            continue
        step = int(m.group(1))
        for k in keys:
            km = re.search(re.escape(k) + r':([-\d.e]+)', line)
            if km:
                steps[k].append(step); out[k].append(float(km.group(1)))
    return steps, out

logs = {'GRPO': 'logs/grpo_musique_seed0_smoke.log', 'VPO': 'logs/vpo_musique_seed0_smoke.log'}
VAL_KEY = 'val-core/musique/reward/best@3/mean'

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for name, color in [('GRPO', 'C3'), ('VPO', 'C0')]:
    s, v = parse_log(logs[name], ['critic/score/mean'])
    ax = axes[0, 0] if name == 'GRPO' else axes[0, 1]
    ax.plot(s['critic/score/mean'], v['critic/score/mean'], color=color)
    ax.set_title(f'{name}: training reward'); ax.set_xlabel('step'); ax.grid(alpha=0.3)

s, v = parse_log(logs['VPO'], ['vpo/own_pool_expected_max_mean'])
axes[1, 0].plot(s['vpo/own_pool_expected_max_mean'], v['vpo/own_pool_expected_max_mean'], color='C0')
axes[1, 0].set_title('VPO objective: E over preferences of best attempt')
axes[1, 0].set_xlabel('step'); axes[1, 0].grid(alpha=0.3)

for name, color in [('GRPO', 'C3'), ('VPO', 'C0')]:
    s, v = parse_log(logs[name], [VAL_KEY])
    axes[1, 1].plot(s[VAL_KEY], v[VAL_KEY], 'o-', color=color, label=name)
axes[1, 1].set_title('validation: best of 3 responses, reward')
axes[1, 1].set_xlabel('step'); axes[1, 1].grid(alpha=0.3); axes[1, 1].legend()
plt.tight_layout(); plt.show()

## 8. Held-out evaluation: 30 answers per question

Now the part the paper's figure is made of. For each of 200 held-out questions we collect **30 answers
per model** under an equal sampling budget:

- **VPO model**: 10 sampled responses, each containing m = 3 attempts (it was trained to write 3).
- **GRPO model**: 30 sampled single-answer responses.
- **Base model** (untrained Qwen3-1.7B): 30 single-answer responses, as a floor.

Every answer goes through the 5-number verifier, giving each model a score tensor of shape
`(200 questions, 10 chains, 3 attempts, 5 objectives)`. The eval script saves that tensor to disk, and
the next section turns it into the best@k curve. Each run takes roughly 15 minutes on one GPU.

In [ ]:
def merged_ckpt(method):
    """Path to the final merged checkpoint of a run (resolves the last saved step)."""
    root = f'checkpoints/vpo_musique/{method}_musique_seed0_smoke'
    step = open(f'{root}/latest_checkpointed_iteration.txt').read().strip()
    return f'{root}/global_step_{step}/actor/huggingface_merged'

!python eval/eval_musique.py --model Qwen/Qwen3-1.7B --method grpo \
    --data-dir ~/data/musique_smoke --num-examples 200 \
    --output results/eval_musique_base.json | tail -15

In [ ]:
!python eval/eval_musique.py --model {merged_ckpt('grpo')} --method grpo \
    --data-dir ~/data/musique_smoke --num-examples 200 \
    --output results/eval_musique_grpo.json | tail -15

In [ ]:
!python eval/eval_musique.py --model {merged_ckpt('vpo')} --method vpo \
    --data-dir ~/data/musique_smoke --num-examples 200 \
    --output results/eval_musique_vpo.json | tail -15

## 9. best@k: the paper's headline metric

The question best@k answers: **if you let a model try k times and keep the best answer, how good is
that best answer on average?** At k = 1 it is plain single-try quality. As k grows it rewards a model
whose tries are *usefully different* from each other: thirty copies of the same wrong answer gain
nothing from more tries, while thirty diverse attempts keep finding new wins.

This is exactly where VPO should earn its keep. GRPO pushes every sample toward the single
highest-scalar behavior, so its 30 answers tend to repeat. VPO was trained to make its attempts cover
different ways of being good, so more tries keep helping.

How we compute it without resampling noise: for one question we have n = 30 scored answers. Instead of
drawing random subsets of size k, there is a closed form for the average of the maximum over **all**
subsets of size k: sort the scores, and weight the j-th smallest by the probability that it is the
largest of a random k-subset, which is C(j-1, k-1) / C(n, k). That is an unbiased estimate of best@k,
the same estimator the paper uses (the same trick as the standard unbiased pass@k formula). We apply it
to the `answer_f1` dimension, then average over the 200 questions.

In [ ]:
import json, math
import numpy as np

def best_at_k(scores, k):
    """Average of max over all k-subsets of scores (exact, no sampling)."""
    s = np.sort(np.asarray(scores, dtype=np.float64))
    n = s.size
    if k == n:
        return float(s[-1])
    log_cnk = math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)
    j = np.arange(k, n + 1)
    lg = np.vectorize(math.lgamma)
    logw = (lg(j) - math.lgamma(k) - lg(j - k + 1)) - log_cnk
    return float((s[k - 1:] * np.exp(logw)).sum())

F1_DIM = -1   # answer_f1 is the last of the 5 objectives
ks = list(range(1, 31))
curves = {}
for label, path in [('Base', 'results/eval_musique_base.json'),
                    ('GRPO', 'results/eval_musique_grpo.json'),
                    ('VPO', 'results/eval_musique_vpo.json')]:
    meta = json.load(open(path))
    t = np.load(meta['tensor_path'])               # (200, 10, 3, 5)
    pool = t.reshape(t.shape[0], -1, t.shape[-1])  # (200, 30, 5)
    f1 = pool[:, :, F1_DIM]                        # (200, 30)
    per_q = np.array([[best_at_k(f1[i], k) for k in ks] for i in range(f1.shape[0])])
    curves[label] = (per_q.mean(0), per_q.std(0) / np.sqrt(per_q.shape[0]))

fig, ax = plt.subplots(figsize=(7, 4.5))
for label, color in [('Base', 'C7'), ('GRPO', 'C3'), ('VPO', 'C0')]:
    mean, se = curves[label]
    ax.plot(ks, mean, color=color, lw=2, label=label)
    ax.fill_between(ks, mean - se, mean + se, color=color, alpha=0.2)
ax.set_xlabel('k (answers kept)'); ax.set_ylabel('best@k answer F1')
ax.set_title('MuSiQue: best answer out of k tries')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

for label in ['Base', 'GRPO', 'VPO']:
    mean, _ = curves[label]
    print(f"{label:5s}  best@1 {mean[0]:.3f}   best@5 {mean[4]:.3f}   best@30 {mean[-1]:.3f}")

## 10. Reading the plot

What to look for, and what the paper-scale runs show:

- **At k = 1** the two trained models should be close. VPO does not pay for its diversity with
  single-try quality.
- **As k grows** the GRPO curve flattens early: its samples concentrate on one behavior, so extra tries
  mostly repeat it. The VPO curve keeps climbing, because its attempts were trained to be usefully
  different, and the gap widens with k.
- **Both should clear the base model** everywhere.

Numbers from this notebook's smoke-scale run will be noisier and lower than the paper's (LoRA, 1/10th
of the training data, 1/4 the group size), but the ordering and the widening gap are the result. For
the full-scale version, run the same two commands on a multi-GPU node with the defaults:

```bash
bash train.sh METHOD=vpo  TASK=musique   # 4x H100, full dataset, batch 128, n=8
bash train.sh METHOD=grpo TASK=musique
```

See [docs/reproducibility.md](../docs/reproducibility.md) for the complete paper configuration.

## Where to go next

- [`01_vpo_from_scratch.ipynb`](01_vpo_from_scratch.ipynb): the algorithm built from nothing, if you
  skipped it.
- [`02_advantage_explorer.ipynb`](02_advantage_explorer.ipynb): poke at the real `vpo_advantage`
  function on synthetic score matrices, no GPU needed.
- `vpo_tasks/musique.py`: everything task-specific you saw here (verifier, prompts, eval) lives in this one
  file. Adding your own task is one file like it plus one import line in `vpo_tasks/__init__.py`.